# Satellite Land Cover Classification: ConvNeXt with Squeeze-and-Excitation Attention on EuroSAT

This notebook implements an end-to-end computer vision pipeline for multi-class satellite earth observation land cover classification on the **EuroSAT** dataset (`torchvision.datasets.EuroSAT`) using a custom **ConvNeXt** architecture with inverted bottleneck blocks and Squeeze-and-Excitation (SE) channel attention.

### Functional Workflow Stages:
1. **Environment Setup & Dependency Installation**
2. **Telemetry Dashboard Extension (`%load_ext tensorboard`)**
3. **Core Library Imports, Deterministic Seeding & SafeTensors Serialization**
4. **Compute Device Allocation (Cloud TPU / CUDA / MPS / CPU)**
5. **Hyperparameter & Directory Configuration**
6. **Vision Data Augmentation Pipeline**
7. **Dataset Ingestion & DataLoader Instantiation**
8. **Exploratory Visual Batch Inspection & Class Distribution**
9. **ConvNeXt Neural Architecture Definition**
10. **Model Instantiation & Parameter Audit**
11. **Loss Formulation & Learning Rate Schedule**
12. **Training & Validation Loop Execution**
13. **Quantitative Evaluation, Confusion Matrix & Grad-CAM Visual Attribution**
14. **Interactive Inference Pipeline**


## 1. Environment Setup and Dependency Installation

Install core runtime dependencies including PyTorch, Torchvision, accelerator runtime bindings, telemetry tooling, and visualization libraries.


In [ ]:
%pip install -q torch torchvision torch_xla tensorboard matplotlib pandas scikit-learn seaborn safetensors ipywidgets

## 2. Telemetry Dashboard Extension

Load the inline TensorBoard viewer to monitor real-time loss trajectories, top-1/top-5 validation accuracy curves, learning rate dynamics, and computational graphs.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir ../../runs

## 3. Core Library Imports, Deterministic Seeding & SafeTensors Serialization

Import scientific computing, image processing, deep learning modules, and configure deterministic random seeds across all engines.


In [ ]:
import json
import random
import time
import zipfile
from datetime import datetime, timezone, timedelta
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
import torchvision
from torchvision import datasets, transforms
from safetensors.torch import save as safetensors_save, load as safetensors_load

try:
    import torch_xla  # type: ignore[import-not-found, import-untyped]
    import torch_xla.core.xla_model as xm  # type: ignore[import-not-found, import-untyped]
except ImportError:
    torch_xla = None
    xm = None


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    if xm is not None:
        try:
            xm.set_rng_state(seed)
        except Exception:
            pass


seed_everything(42)
print(
    f"PyTorch Version: {torch.__version__} | Torchvision Version: {torchvision.__version__}"
)


def save_model_zip(
    model: torch.nn.Module,
    metadata: dict,
    save_path: Path,
    loss_fn: torch.nn.Module | None = None,
    is_xla: bool = False,
) -> None:
    """Saves model weights as SafeTensors and metadata as JSON within a zip archive."""
    save_path.parent.mkdir(parents=True, exist_ok=True)
    state_dict = {
        k: v.detach().cpu().contiguous() for k, v in model.state_dict().items()
    }
    tensor_bytes = safetensors_save(state_dict)

    metadata_clean = {}
    for k, v in metadata.items():
        if isinstance(v, (int, float, str, bool, list, dict)) or v is None:
            metadata_clean[k] = v
        elif hasattr(v, "item"):
            metadata_clean[k] = v.item()
        elif hasattr(v, "tolist"):
            metadata_clean[k] = v.tolist()
        else:
            metadata_clean[k] = str(v)

    meta_bytes = json.dumps(metadata_clean, indent=2).encode("utf-8")

    with zipfile.ZipFile(save_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("model.safetensors", tensor_bytes)
        zf.writestr("metadata.json", meta_bytes)
        if loss_fn is not None and len(loss_fn.state_dict()) > 0:
            loss_dict = {
                k: v.detach().cpu().contiguous()
                for k, v in loss_fn.state_dict().items()
            }
            zf.writestr("loss_fn.safetensors", safetensors_save(loss_dict))


def load_model_zip(
    model: torch.nn.Module,
    load_path: Path,
    device: torch.device,
    loss_fn: torch.nn.Module | None = None,
) -> dict:
    """Loads model weights from SafeTensors inside zip archive and returns metadata dictionary."""
    with zipfile.ZipFile(load_path, "r") as zf:
        tensor_bytes = zf.read("model.safetensors")
        meta_bytes = zf.read("metadata.json")
        if loss_fn is not None and "loss_fn.safetensors" in zf.namelist():
            loss_bytes = zf.read("loss_fn.safetensors")
            loss_dict = safetensors_load(loss_bytes)
            loss_dict = {k: v.to(device) for k, v in loss_dict.items()}
            loss_fn.load_state_dict(loss_dict)

    state_dict = safetensors_load(tensor_bytes)
    state_dict = {k: v.to(device) for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    metadata = json.loads(meta_bytes.decode("utf-8"))
    return metadata

## 4. Compute Device Allocation

Resolve compute accelerator dynamically with priority order: Cloud TPU (PyTorch XLA) -> NVIDIA CUDA -> Apple Silicon MPS -> Multi-core CPU.


In [ ]:
device = None
device_name = None

# 1. Prioritize Cloud TPU
if xm is not None:
    try:
        xla_dev = xm.xla_device()
        if xm.xla_device_hw(xla_dev) == "TPU":
            device = xla_dev
            device_name = "Cloud TPU (PyTorch XLA)"
    except (RuntimeError, ValueError) as exc:
        print(f"Notice: TPU initialization failed ({exc}). Falling back to GPU/CPU.")

# 2. Prioritize GPU (NVIDIA CUDA or Apple Silicon MPS)
if device is None and torch.cuda.is_available():
    device = torch.device("cuda")
    device_name = f"NVIDIA GPU ({torch.cuda.get_device_name(0)})"
elif (
    device is None
    and hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    device = torch.device("mps")
    device_name = "Apple Silicon GPU (MPS)"

# 3. Fallback to CPU
if device is None:
    device = torch.device("cpu")
    device_name = "CPU"

is_xla = device.type == "xla"
print(f"Allocated Compute Device: {device} [{device_name}]")

## 5. Hyperparameter and Directory Configuration

Establish paths for dataset cache, runs, and model checkpoints, as well as optimization hyperparameters.


In [ ]:
# Directories
DATA_DIR = Path("../../data").resolve()
RUN_TIMESTAMP = datetime.now(tz=timezone(timedelta(hours=1))).strftime(
    "%Y-%m-%d_%H-%M-%S"
)
LOG_DIR = Path("../../runs/vision_eurosat_convnext").resolve() / RUN_TIMESTAMP
MODEL_DIR = Path("../../models/vision").resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "eurosat_convnext.zip"

# Hyperparameters
BATCH_SIZE = 128
NUM_EPOCHS = 15
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
NUM_CLASSES = 10
IMAGE_SIZE = 64
NUM_WORKERS = 2

EUROSAT_CLASSES = [
    "AnnualCrop",
    "Forest",
    "HerbaceousVegetation",
    "Highway",
    "Industrial",
    "Pasture",
    "PermanentCrop",
    "Residential",
    "River",
    "SeaLake",
]

print(f"Dataset Dir : {DATA_DIR}")
print(f"Log Dir     : {LOG_DIR}")
print(f"Model Path  : {MODEL_PATH}")

## 6. Vision Data Augmentation Pipeline

Define train and validation transform pipelines including random horizontal/vertical flips, affine rotations, color jitter, and standard Sentinel-2/ImageNet normalization.


In [ ]:
# EuroSAT Mean and Standard Deviation (RGB channels)
EUROSAT_MEAN = [0.485, 0.456, 0.406]
EUROSAT_STD = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose(
    [
        transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=EUROSAT_MEAN, std=EUROSAT_STD),
    ]
)

val_transforms = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=EUROSAT_MEAN, std=EUROSAT_STD),
    ]
)

## 7. Dataset Ingestion & DataLoader Instantiation

Download and load EuroSAT dataset, splitting into training (80%), validation (10%), and test (10%) sets.


In [ ]:
# Ingest raw dataset
full_dataset = datasets.EuroSAT(root=DATA_DIR, download=True)
total_samples = len(full_dataset)
print(f"Total EuroSAT Samples: {total_samples}")

# 80 / 10 / 10 Split
train_size = int(0.80 * total_samples)
val_size = int(0.10 * total_samples)
test_size = total_samples - train_size - val_size

train_subset, val_subset, test_subset = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42),
)


class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, target = self.subset[idx]
        if self.transform is not None:
            img = self.transform(img)
        return img, target


train_dataset = TransformSubset(train_subset, transform=train_transforms)
val_dataset = TransformSubset(val_subset, transform=val_transforms)
test_dataset = TransformSubset(test_subset, transform=val_transforms)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

print(
    f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}"
)

## 8. Exploratory Visual Batch Inspection

Visualize a grid of sample satellite tiles with ground-truth land cover category labels.


In [ ]:
def unnormalize(tensor, mean=EUROSAT_MEAN, std=EUROSAT_STD):
    img = tensor.clone().cpu().numpy().transpose(1, 2, 0)
    img = img * np.array(std) + np.array(mean)
    return np.clip(img, 0, 1)


sample_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
images, labels = next(iter(sample_loader))

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for idx, ax in enumerate(axes.flat):
    img_np = unnormalize(images[idx])
    ax.imshow(img_np)
    ax.set_title(EUROSAT_CLASSES[labels[idx].item()], fontsize=9)
    ax.axis("off")
plt.suptitle("EuroSAT Sentinel-2 Satellite Tiles Sample Batch", fontsize=14, y=0.92)
plt.tight_layout()
plt.show()

## 9. ConvNeXt Neural Architecture Definition

Implement a modern **ConvNeXt** architecture with depthwise separable 7x7 convolutions, LayerNorm, GELU, inverted bottleneck expansion, and Squeeze-and-Excitation (SE) channel attention gating.


In [ ]:
class SqueezeExcitation(nn.Module):
    """Channel attention module."""

    def __init__(self, channels: int, reduction: int = 4):
        super().__init__()
        reduced = max(channels // reduction, 8)
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, reduced, bias=False),
            nn.GELU(),
            nn.Linear(reduced, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _, _ = x.shape
        w = self.fc(x).view(b, c, 1, 1)
        return x * w


class ConvNeXtBlock(nn.Module):
    """ConvNeXt block with 7x7 Depthwise Conv, LayerNorm, Inverted Bottleneck, GELU and SE."""

    def __init__(self, dim: int, expansion: int = 4):
        super().__init__()
        self.dwconv = nn.Conv2d(
            dim, dim, kernel_size=7, padding=3, groups=dim, bias=False
        )
        self.norm = nn.GroupNorm(1, dim)  # Equivalent to LayerNorm across channels
        self.pwconv1 = nn.Conv2d(dim, dim * expansion, kernel_size=1)
        self.act = nn.GELU()
        self.pwconv2 = nn.Conv2d(dim * expansion, dim, kernel_size=1)
        self.se = SqueezeExcitation(dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        out = self.dwconv(x)
        out = self.norm(out)
        out = self.pwconv1(out)
        out = self.act(out)
        out = self.pwconv2(out)
        out = self.se(out)
        return residual + out


class ConvNeXtSatellite(nn.Module):
    """Custom ConvNeXt classifier for high-resolution satellite land cover analysis."""

    def __init__(
        self,
        in_channels: int = 3,
        num_classes: int = 10,
        depths=[2, 2, 6, 2],
        dims=[32, 64, 128, 256],
    ):
        super().__init__()
        # Stem
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, dims[0], kernel_size=4, stride=2, padding=1),
            nn.GroupNorm(1, dims[0]),
        )
        # Stages
        self.stages = nn.ModuleList()
        for i in range(len(depths)):
            stage_blocks = [ConvNeXtBlock(dims[i]) for _ in range(depths[i])]
            if i < len(depths) - 1:
                # Downsampling layer between stages
                stage_blocks.append(
                    nn.Sequential(
                        nn.GroupNorm(1, dims[i]),
                        nn.Conv2d(dims[i], dims[i + 1], kernel_size=2, stride=2),
                    )
                )
            self.stages.append(nn.Sequential(*stage_blocks))

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.LayerNorm(dims[-1]),
            nn.Dropout(0.2),
            nn.Linear(dims[-1], num_classes),
        )

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        for stage in self.stages:
            x = stage(x)
        return x

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.forward_features(x)
        logits = self.head(features)
        return logits

## 10. Model Instantiation and Parameter Summary

Instantiate the network on the target accelerator and compute total trainable parameters.


In [ ]:
model = ConvNeXtSatellite(in_channels=3, num_classes=NUM_CLASSES).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"ConvNeXt Satellite Architecture Summary:")
print(f" - Total Parameters     : {total_params:,}")
print(f" - Trainable Parameters : {trainable_params:,}")

# Sanity forward pass
dummy_input = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
with torch.no_grad():
    dummy_out = model(dummy_input)
print(f" - Output Logits Shape  : {dummy_out.shape} (Expected: [2, {NUM_CLASSES}])")

## 11. Loss Formulation, Optimizer & Learning Rate Scheduling

Configure Label-Smoothed Cross-Entropy Loss, AdamW optimizer with weight decay, and OneCycleLR scheduler.


In [ ]:
loss_fn = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
total_steps = NUM_EPOCHS * len(train_loader)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LEARNING_RATE,
    total_steps=total_steps,
    pct_start=0.1,
    anneal_strategy="cos",
)

writer = SummaryWriter(log_dir=str(LOG_DIR))
print(f"Initialized TensorBoard SummaryWriter at: {LOG_DIR}")

## 12. Training and Validation Loop Execution

Execute optimization loop across epochs with progress tracking, TensorBoard telemetry logging, and best-model SafeTensors saving.


In [ ]:
best_val_acc = 0.0
train_history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

print(f"Starting Training for {NUM_EPOCHS} Epochs on {device} [{device_name}]...")

for epoch in range(1, NUM_EPOCHS + 1):
    start_time = time.time()
    # ── Train Step ──────────────────────────────────────────────────────────
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()

        logits = model(images)
        loss = loss_fn(logits, targets)
        loss.backward()

        if is_xla:
            xm.optimizer_step(optimizer)
        else:
            optimizer.step()
        scheduler.step()

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

    epoch_train_loss = running_loss / total
    epoch_train_acc = correct / total * 100.0

    # ── Validation Step ────────────────────────────────────────────────────
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, targets in val_loader:
            images, targets = images.to(device), targets.to(device)
            logits = model(images)
            loss = loss_fn(logits, targets)

            val_loss += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            val_correct += (preds == targets).sum().item()
            val_total += targets.size(0)

    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total * 100.0
    epoch_time = time.time() - start_time

    # Telemetry Logging
    writer.add_scalar("Loss/Train", epoch_train_loss, epoch)
    writer.add_scalar("Loss/Val", epoch_val_loss, epoch)
    writer.add_scalar("Accuracy/Train", epoch_train_acc, epoch)
    writer.add_scalar("Accuracy/Val", epoch_val_acc, epoch)
    writer.add_scalar("LearningRate", optimizer.param_groups[0]["lr"], epoch)
    writer.flush()

    train_history["train_loss"].append(epoch_train_loss)
    train_history["val_loss"].append(epoch_val_loss)
    train_history["train_acc"].append(epoch_train_acc)
    train_history["val_acc"].append(epoch_val_acc)

    checkpoint_msg = ""
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        save_metadata = {
            "epoch": epoch,
            "best_val_acc": best_val_acc,
            "val_loss": epoch_val_loss,
            "architecture": "ConvNeXtSatellite",
            "dataset": "EuroSAT",
            "num_classes": NUM_CLASSES,
            "timestamp": RUN_TIMESTAMP,
        }
        save_model_zip(model, save_metadata, MODEL_PATH, loss_fn=loss_fn, is_xla=is_xla)
        checkpoint_msg = " [★ Best Checkpoint Saved]"

    print(
        f"Epoch [{epoch:02d}/{NUM_EPOCHS:02d}] ({epoch_time:.1f}s) | "
        f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}% | "
        f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}% (Best: {best_val_acc:.2f}%){checkpoint_msg}"
    )

writer.close()
print(f"Training Complete! Peak Validation Accuracy: {best_val_acc:.2f}%")

## 13. Quantitative Evaluation, Confusion Matrix & Grad-CAM Visual Attribution

Load the best saved SafeTensors checkpoint, evaluate on the independent test split, generate precision/recall classification report, confusion matrix, and visualize Grad-CAM activation heatmaps.


In [ ]:
# Load Best Checkpoint
metadata = load_model_zip(model, MODEL_PATH, device=device, loss_fn=loss_fn)
print(
    f"Loaded Best Checkpoint from Epoch {metadata['epoch']} (Val Acc: {metadata['best_val_acc']:.2f}%)"
)

# Test Split Evaluation
model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for images, targets in test_loader:
        images = images.to(device)
        logits = model(images)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(targets.numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)
test_acc = (all_preds == all_targets).mean() * 100.0
print(f"\nFinal Test Accuracy: {test_acc:.2f}%\n")

print("Classification Report:")
print(
    classification_report(
        all_targets, all_preds, target_names=EUROSAT_CLASSES, digits=4
    )
)

# Confusion Matrix Heatmap
cm = confusion_matrix(all_targets, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=EUROSAT_CLASSES,
    yticklabels=EUROSAT_CLASSES,
)
plt.title(f"EuroSAT Confusion Matrix (Test Accuracy: {test_acc:.2f}%)", fontsize=13)
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 14. Interactive Inference Pipeline

Run end-to-end inference on random test samples and visualize class probability distributions.


In [ ]:
def predict_and_visualize(model, dataset, num_samples=6):
    model.eval()
    fig, axes = plt.subplots(num_samples, 2, figsize=(10, 3 * num_samples))
    indices = random.sample(range(len(dataset)), num_samples)

    with torch.no_grad():
        for i, idx in enumerate(indices):
            img_tensor, true_label = dataset[idx]
            img_batch = img_tensor.unsqueeze(0).to(device)
            logits = model(img_batch)
            probs = F.softmax(logits, dim=1).cpu().numpy()[0]
            pred_label = probs.argmax()

            # Show Image
            ax_img = axes[i, 0] if num_samples > 1 else axes[0]
            ax_img.imshow(unnormalize(img_tensor))
            color = "green" if pred_label == true_label else "red"
            ax_img.set_title(
                f"True: {EUROSAT_CLASSES[true_label]}\nPred: {EUROSAT_CLASSES[pred_label]} ({probs[pred_label] * 100:.1f}%)",
                color=color,
            )
            ax_img.axis("off")

            # Show Probabilities
            ax_bar = axes[i, 1] if num_samples > 1 else axes[1]
            y_pos = np.arange(len(EUROSAT_CLASSES))
            ax_bar.barh(y_pos, probs, color="skyblue")
            ax_bar.set_yticks(y_pos)
            ax_bar.set_yticklabels(EUROSAT_CLASSES, fontsize=8)
            ax_bar.set_xlim(0, 1)
            ax_bar.set_xlabel("Confidence Probability")
            ax_bar.invert_yaxis()

    plt.tight_layout()
    plt.show()


predict_and_visualize(model, test_dataset, num_samples=5)